[Project Stone]
- 돌 분류 프로젝트

In [74]:
import sys
import torch
import os

print("--- Environment Check ---")
print(f"Python Executable: {sys.executable}") # 현재 사용 중인 파이썬 실행 파일 경로
print(f"Python Version: {sys.version}")      # 현재 사용 중인 파이썬 버전
print(f"PyTorch Version: {torch.__version__}") # 현재 로드된 PyTorch 버전
print(f"PyTorch CUDA Build: {torch.version.cuda if hasattr(torch.version, 'cuda') else 'N/A'}") # PyTorch가 빌드된 CUDA 버전
print(f"CUDA Available: {torch.cuda.is_available()}") # CUDA 사용 가능 여부
if torch.cuda.is_available():
    print(f"CUDA Version (Runtime): {torch.version.cuda}") # PyTorch가 인식하는 런타임 CUDA 버전
    print(f"Device Name: {torch.cuda.get_device_name(0)}") # GPU 이름
    print(f"Device Compute Capability: {torch.cuda.get_device_capability(0)}") # Compute Capability 확인
print("-" * 25)

--- Environment Check ---
Python Executable: c:\Users\kdt\anaconda3\envs\DL_TORCH\python.exe
Python Version: 3.9.21 (main, Dec 11 2024, 16:35:24) [MSC v.1929 64 bit (AMD64)]
PyTorch Version: 2.4.0
PyTorch CUDA Build: None
CUDA Available: False
-------------------------


In [75]:
import pandas as pd
import numpy as np

In [76]:
numlist = os.listdir('./data/training')
trainDIR = './data/training'
sum = 0

for i in numlist:
    a = (len(os.listdir('./data/training/'+i)))
    print(i, a)
    sum += a
print(sum)

etc 138
n0 105
n1 111
354


In [77]:
## 작업순서
## 데이테 셋 만들기 
## 데이터 로더 만들기
## 폴더별로 되있으니까 imgeafolder사용하면 될듯?


In [78]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from torch.utils.data import Dataset, DataLoader  # Pytorch의 데이터셋 관련
from torchvision import transforms  # 전처리모듈
from torchvision.datasets import ImageFolder

from PIL import Image


In [79]:
TRANSFORM = transforms.Compose([
        transforms.Resize((224, 224)),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])



In [80]:
# classes = ['alouatta_palliata', 'erythrocebus_patas']
classes = ['n0', 'n1']
classes2idx = {x:idx+1 for idx,x in enumerate(classes)}
classes2idx['Etc']= 0
classes2idx

{'n0': 1, 'n1': 2, 'Etc': 0}

In [81]:
class CustomImageFolder(ImageFolder):
    def __init__(self, root, transform=None, classes2idx=classes2idx):
        self.classes2idx = classes2idx
        self.class_to_idx = self.classes2idx
        self.classes = list(classes2idx.keys())
        super().__init__(root, transform)

    def find_classes(self, directory):
        # 이 함수가 자동 클래스 탐색을 담당하는데, 우리가 원하는 대로 덮어씀
        return self.classes, self.classes2idx

In [82]:
os.listdir(trainDIR)

['etc', 'n0', 'n1']

In [83]:
trainDS = CustomImageFolder(trainDIR, transform=TRANSFORM, classes2idx=classes2idx)

In [84]:
for a,b in trainDS:
    print(a,b)
    break

tensor([[[-0.7479, -0.7479, -0.7308,  ...,  0.3481,  0.4508,  0.4679],
         [-0.7137, -0.7137, -0.6965,  ...,  0.0227,  0.0569,  0.1083],
         [-0.6794, -0.6794, -0.6623,  ...,  0.0227,  0.0227,  0.0398],
         ...,
         [-0.0629, -0.0801, -0.1143,  ...,  0.8276,  0.8789,  0.8961],
         [-0.0972, -0.1143, -0.1657,  ...,  0.8276,  0.8789,  0.8961],
         [-0.4397, -0.4568, -0.4397,  ...,  0.2967,  0.3481,  0.3481]],

        [[-1.0553, -1.0553, -1.0378,  ..., -0.3901, -0.3200, -0.3200],
         [-1.0378, -1.0203, -1.0028,  ..., -0.6176, -0.6176, -0.5826],
         [-0.9853, -1.0028, -0.9853,  ..., -0.6176, -0.6527, -0.6527],
         ...,
         [-0.8978, -0.9153, -0.9328,  ..., -0.0574, -0.0049,  0.0301],
         [-0.9328, -0.9328, -0.9678,  ...,  0.0126,  0.0301,  0.0476],
         [-1.0203, -1.0378, -1.0203,  ..., -0.3200, -0.2850, -0.2675]],

        [[-1.4210, -1.4210, -1.4036,  ..., -1.6999, -1.6999, -1.7173],
         [-1.3687, -1.3513, -1.3513,  ..., -1

In [85]:
print('train', len(trainDS))
print(trainDS.classes)
trainDS.class_to_idx
idx2class = {values:key for key, values in classes2idx.items()}
idx2class

train 354
['n0', 'n1', 'Etc']


{1: 'n0', 2: 'n1', 0: 'Etc'}

In [86]:
cnum = 0
for (a, b) in trainDS:
    cnum += 1
    print(a[0].shape, idx2class[b])
    if cnum == 10: break
    

torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc
torch.Size([224, 224]) Etc


In [87]:
from torch.utils.data import random_split

# 전체 데이터 수
total_size = len(trainDS)
train_size = int(0.8 * total_size)
valid_size = total_size - train_size

# 무작위로 train/valid 분리
trainDS, validDS = random_split(trainDS, [train_size, valid_size])

In [88]:
print(len(trainDS))
print(len(validDS))

283
71


In [89]:
imgTS, label = trainDS[0]   ## __getitem__(index)
print(imgTS.shape, label)


torch.Size([3, 224, 224]) 1


In [90]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

In [91]:
# plt.imshow(imgTS.permute(1,2,0))
# plt.title(idx2class[label])
# plt.show()

In [92]:
def collator(batch):
    images, labels = zip(*batch)  # 튜플 of Tensors
    images = torch.stack(images)  # → Tensor of shape [B, C, H, W]
    labels = torch.tensor(labels) # → Tensor of shape [B]
    return images, labels

BATCH_SIZE = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 0.0001
DEVICE

'cpu'

In [93]:
trainDL = DataLoader(
    trainDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)
validDL = DataLoader(
    validDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)

In [94]:
for a, b in trainDL:
    print((type(a)),(type(b)))
    break

<class 'torch.Tensor'> <class 'torch.Tensor'>


In [95]:
from torchvision import models
from torchvision import ops
from torchvision.models.detection import rpn

num_classes = len(classes2idx)
num_classes

3

In [96]:
# ✅ 모델: ResNet101 + 마지막 fc 교체
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)


c:\Users\kdt\anaconda3\envs\DL_TORCH\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\kdt\anaconda3\envs\DL_TORCH\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [97]:
from torch import optim
from tqdm import tqdm
# ✅ 손실함수, 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


In [98]:
DEVICE

'cpu'

In [99]:
torch.__version__

'2.4.0'

In [100]:
# %conda install pytorch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 pytorch-cuda=12.4 -c pytorch -c nvidia

In [101]:
# %pip install psutil
import psutil
import subprocess
from tqdm import tqdm

def get_gpu_usage():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,nounits,noheader"]
        )
        result = result.decode("utf-8").strip().split("\n")[0]
        gpu_util, mem_used, mem_total = map(int, result.split(", "))
        return gpu_util, mem_used, mem_total
    except Exception:
        return None, None, None

In [102]:
EPOCH = 5
best_val_loss = float('inf')
patience = 3
patience_counter = 0

train_metrics = []
val_metrics = []

for epoch in range(EPOCH):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    epoch_train_cpu = []
    epoch_train_gpu = []
    epoch_train_loss = []
    epoch_train_acc = []

    train_loop = tqdm(trainDL, desc=f"[Epoch {epoch+1}] Training")
    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        acc = 100. * correct / total

        # Metric 측정
        cpu_usage = psutil.cpu_percent(interval=None)
        gpu_util, mem_used, mem_total = get_gpu_usage()

        # 저장
        epoch_train_cpu.append(cpu_usage)
        epoch_train_gpu.append(gpu_util if gpu_util is not None else 0)
        epoch_train_loss.append(loss.item())
        epoch_train_acc.append(acc)

        train_loop.set_postfix(loss=loss.item(), acc=acc, cpu=f"{cpu_usage}%", gpu=f"{gpu_util}%")

    train_metrics.append({
        "cpu": epoch_train_cpu,
        "gpu": epoch_train_gpu,
        "loss": epoch_train_loss,
        "acc": epoch_train_acc
    })

    train_acc = 100. * correct / total
    train_loss = running_loss / len(trainDL)

    # ----------- Validation -----------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    epoch_val_cpu = []
    epoch_val_gpu = []
    epoch_val_loss = []
    epoch_val_acc = []

    val_loop = tqdm(validDL, desc=f"[Epoch {epoch+1}] Validation")
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)

            acc = 100. * val_correct / val_total

            cpu_usage = psutil.cpu_percent(interval=None)
            gpu_util, mem_used, mem_total = get_gpu_usage()

            # 저장
            epoch_val_cpu.append(cpu_usage)
            epoch_val_gpu.append(gpu_util if gpu_util is not None else 0)
            epoch_val_loss.append(loss.item())
            epoch_val_acc.append(acc)

            val_loop.set_postfix(loss=loss.item(), acc=acc, cpu=f"{cpu_usage}%", gpu=f"{gpu_util}%")

    val_metrics.append({
        "cpu": epoch_val_cpu,
        "gpu": epoch_val_gpu,
        "loss": epoch_val_loss,
        "acc": epoch_val_acc
    })

    val_acc = 100. * val_correct / val_total
    val_loss_avg = val_loss / len(validDL)

    if val_loss_avg < best_val_loss and val_loss_avg < 2:
        best_val_loss = val_loss_avg
        torch.save(model.state_dict(), f"./_model/{epoch}_vl{val_loss_avg:.2f}.pth")

    print(f"\n[Epoch {epoch+1}] Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"                    Valid Loss: {val_loss_avg:.4f}, Valid Acc: {val_acc:.2f}%")

# -------------------------------
# 모든 metric을 담은 리스트
# -------------------------------
# 예: train_metrics[0]["loss"] → 첫 번째 에포크의 모든 배치 loss
#     val_metrics[0]["gpu"]  → 첫 번째 에포크의 모든 배치 GPU 사용률

[Epoch 1] Validation: 100%|██████████| 7/7 [00:04<00:00,  1.42it/s, acc=97.1, cpu=55.7%, gpu=None%, loss=0.0205]



[Epoch 1] Train Loss: 0.4298, Train Acc: 85.71%
                    Valid Loss: 0.1430, Valid Acc: 97.14%


[Epoch 2] Validation: 100%|██████████| 7/7 [00:06<00:00,  1.16it/s, acc=97.1, cpu=59.9%, gpu=None%, loss=0.0276]



[Epoch 2] Train Loss: 0.1108, Train Acc: 96.79%
                    Valid Loss: 0.1457, Valid Acc: 97.14%


[Epoch 3] Validation: 100%|██████████| 7/7 [00:06<00:00,  1.14it/s, acc=95.7, cpu=58.6%, gpu=None%, loss=0.0929]



[Epoch 3] Train Loss: 0.1530, Train Acc: 95.36%
                    Valid Loss: 0.1860, Valid Acc: 95.71%


[Epoch 4] Validation: 100%|██████████| 7/7 [00:05<00:00,  1.38it/s, acc=95.7, cpu=52.7%, gpu=None%, loss=0.0271]



[Epoch 4] Train Loss: 0.1769, Train Acc: 95.00%
                    Valid Loss: 0.1416, Valid Acc: 95.71%


[Epoch 5] Validation: 100%|██████████| 7/7 [00:06<00:00,  1.02it/s, acc=97.1, cpu=56.1%, gpu=None%, loss=0.505] 


[Epoch 5] Train Loss: 0.0412, Train Acc: 98.93%
                    Valid Loss: 0.0882, Valid Acc: 97.14%


In [103]:
class TestImageDataset(Dataset):
    def __init__(self, csv_df, image_root, transform=None):
        self.df = csv_df
        self.image_root = image_root  # 예: './_data/open/test/'
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        img_path = self.df.iloc[idx, 1]
        img_path = os.path.join(self.image_root, img_path)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name


In [104]:
TRANSFORM_T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
])



In [105]:
testDS = ImageFolder("./_data/test", transform=TRANSFORM_T)
testDL = DataLoader(testDS, batch_size=32, shuffle=False)

FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: './_data/test'

In [ ]:
for a, b in testDS:
    print(a,b)
    break

tensor([[[2.0948, 2.0948, 2.0948,  ..., 2.1290, 2.1462, 2.1462],
         [2.0948, 2.0948, 2.0948,  ..., 2.0948, 2.1119, 2.1119],
         [2.0948, 2.0948, 2.0948,  ..., 2.1119, 2.1290, 2.1290],
         ...,
         [2.1633, 2.1462, 2.1290,  ..., 2.1290, 2.1290, 2.1462],
         [2.1633, 2.1462, 2.1290,  ..., 2.1290, 2.1462, 2.1633],
         [2.1633, 2.1462, 2.1290,  ..., 2.1462, 2.1462, 2.1633]],

        [[2.3060, 2.3060, 2.3060,  ..., 2.3060, 2.3235, 2.3235],
         [2.3060, 2.3060, 2.3060,  ..., 2.2710, 2.2885, 2.2885],
         [2.3060, 2.3060, 2.3060,  ..., 2.2885, 2.3060, 2.3060],
         ...,
         [2.3410, 2.3235, 2.3060,  ..., 2.3060, 2.3060, 2.3235],
         [2.3585, 2.3235, 2.3060,  ..., 2.3060, 2.3235, 2.3410],
         [2.3585, 2.3235, 2.3060,  ..., 2.3235, 2.3235, 2.3410]],

        [[2.5006, 2.5006, 2.5006,  ..., 2.5180, 2.5354, 2.5354],
         [2.5006, 2.5006, 2.5006,  ..., 2.4831, 2.5006, 2.5006],
         [2.5006, 2.5006, 2.5006,  ..., 2.5006, 2.5180, 2.

In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    count = 0
    for images, filenames in testDL:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        
        for fname, pred in zip(filenames, preds):
            predictions.append((fname, idx2class[pred.item()]))
        count += 1
        # if count ==3: break

In [ ]:
predictions

[(tensor(0), 'Etc'),
 (tensor(0), 'erythrocebus_patas'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'erythrocebus_patas'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'erythrocebus_patas'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'erythrocebus_patas'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'erythrocebus_patas'),
 (tensor(0), 'erythrocebus_patas'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'alouatta_palliata'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'erythrocebus_patas'),
 (tensor(0), 'alouatta_palliata'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc'),
 (tensor(0), 'erythrocebus_patas'),
 (tensor(0), 'Etc'),
 (tensor(0), 'alouatta_palliata'),
 (tensor(0), 'Etc'),
 (tensor(0), 'Etc')

In [ ]:
predictions

In [ ]:
sampleDF.to_csv('./_data/open/sample_submission_answer.csv')